# Fase 3 · Pipeline y modelado con PySpark

**Proyecto integrador de tecnologías de Big Data** · Predicción de delitos en Chicago · Valentino De Paola, Eduardo López y Carlos David González

> **Pregunta de la Fase 2:** ¿En qué zonas, días y horarios de Chicago es más probable que ocurra un delito en las próximas semanas, y qué tipo de delito será?

Este notebook ejecuta el plan CRISP-DM de la Fase 2 sobre el clúster (HDFS + Spark sobre YARN):

- **A. Preparación de datos** (CRISP-DM III): carga, limpieza, transformación y guardado en Parquet.
- **B. Modelado analítico** (CRISP-DM IV): dos modelos de SparkML y su comparación.
- **C. Visualización:** cinco gráficas que responden las preguntas analíticas.

**Cómo correrlo:** en la instancia, con `LOCAL = False`, *Kernel → Restart Kernel and Run All Cells*. El CSV crudo lo deja en HDFS `scripts/ingesta-raw.sh`, que imprime la fecha que va en `FECHA_RAW`.

## Configuración

In [1]:
LOCAL = False                 # False: instancia (HDFS + YARN) · True: prueba en una compu con el CSV local
FECHA_RAW = "2026-10-23"      # carpeta de la ingesta en HDFS (la imprime scripts/ingesta-raw.sh)

RUTA_RAW_LOCAL = "../data/Crimes_2025.csv"
RUTA_RAW_HDFS = f"hdfs:///datalake/raw/crimes_2025/fecha={FECHA_RAW}/Crimes_2025.csv"
RUTA_RAW = RUTA_RAW_LOCAL if LOCAL else RUTA_RAW_HDFS

In [2]:
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

builder = (SparkSession.builder
           .appName(f"fase3-pipeline-{'local' if LOCAL else 'cluster'}")
           .config("spark.sql.session.timeZone", "UTC"))

if LOCAL:
    spark = (builder.master("local[*]")
             .config("spark.driver.memory", "2g")
             .config("spark.ui.showConsoleProgress", "false")
             .getOrCreate())
else:
    spark = builder.enableHiveSupport().getOrCreate()   # master = yarn (spark-defaults.conf)

spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| master:", spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/23 19:16:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0 | master: yarn


# A. Preparación de datos con PySpark · CRISP-DM Fase III

## A1. Carga e inspección inicial

### Paso 1 · Carga desde HDFS

Se lee el CSV crudo tal como lo entrega el portal de datos de Chicago, con `inferSchema=True` para ver qué tipos detecta Spark por su cuenta. Los tipos definitivos (por ejemplo, convertir `Date` de texto a fecha) se fijan en A2.

In [3]:
df_raw = spark.read.csv(RUTA_RAW, header=True, inferSchema=True)
print("Leído desde:", RUTA_RAW)

[Stage 1:=============================>                             (1 + 1) / 2]

Leído desde: hdfs:///datalake/raw/crimes_2025/fecha=2026-10-23/Crimes_2025.csv


### Paso 2 · Exploración rápida

Esquema, tamaño y estadísticas descriptivas.

In [4]:
df_raw.printSchema()

filas = df_raw.count()
print(f"Filas: {filas:,} · Columnas: {len(df_raw.columns)}")

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



[Stage 2:>                                                          (0 + 2) / 2]

Filas: 238,174 · Columnas: 22


Estadísticas de las columnas numéricas. Ojo: de ellas, solo las coordenadas son medidas de verdad; `ID`, `Beat`, `District`, `Ward`, `Community Area` y `Year` son códigos que Spark lee como números, así que su media o desviación no tienen sentido.

In [5]:
numericas = [c.name for c in df_raw.schema.fields if isinstance(c.dataType, NumericType)]
estadisticas = (df_raw.select(numericas).describe().toPandas()
                .set_index("summary").T.apply(pd.to_numeric).round(2))
estadisticas

summary,count,mean,stddev,min,max
ID,238174,13869595.01,602434.00,28545.00,14366259.00
Beat,238174,1151.95,730.89,111.00,6100.00
District,238174,11.29,7.30,1.00,61.00
Ward,238173,23.05,13.96,0.00,50.00
Community Area,238165,36.29,21.57,0.00,77.00
X Coordinate,235484,1165672.31,16268.59,1093568.00,1205119.00
Y Coordinate,235484,1887177.31,31434.69,1813897.00,1951517.00
Year,238174,2025.00,0.00,2025.00,2025.00
Latitude,235484,41.85,0.09,41.64,42.02
Longitude,235484,-87.67,0.06,-87.93,-87.52


Cuántos valores distintos tiene cada variable categórica clave, y los tipos de delito más frecuentes.

In [6]:
categoricas = ["Primary Type", "Description", "Location Description", "IUCR", "FBI Code",
               "Beat", "District", "Ward", "Community Area"]
distintos = (df_raw.agg(*[F.countDistinct(c).alias(c) for c in categoricas])
             .toPandas().T.rename(columns={0: "valores distintos"}))
distintos

,valores distintos
Primary Type,31
Description,315
Location Description,130
IUCR,337
FBI Code,26
Beat,275
District,24
Ward,51
Community Area,78


In [7]:
top_tipos = (df_raw.groupBy("Primary Type").count()
             .orderBy(F.desc("count")).limit(10)
             .withColumn("% del total", F.round(F.col("count") * 100 / filas, 2))
             .toPandas())
top_tipos

,Primary Type,count,% del total
0,THEFT,55201,23.18
1,BATTERY,42667,17.91
2,CRIMINAL DAMAGE,26249,11.02
3,ASSAULT,21601,9.07
4,MOTOR VEHICLE THEFT,17261,7.25
5,OTHER OFFENSE,16863,7.08
6,DECEPTIVE PRACTICE,15370,6.45
7,BURGLARY,9743,4.09
8,NARCOTICS,7420,3.12
9,ROBBERY,5816,2.44


### Paso 3 · Tabla de variables

Para cada variable: tipo que detectó Spark, nulos (vacíos incluidos) en cantidad y porcentaje, y observaciones. Las observaciones combinan lo que se mide aquí con las decisiones del plan de la Fase 2.

In [8]:
# Nulos y vacíos de todas las columnas en una sola pasada
nulos = df_raw.agg(*[
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
    for c in df_raw.columns
]).first().asDict()

# Mediciones que se citan en las observaciones
hora_exacta = df_raw.filter(F.col("Date").endswith(" 12:00:00 AM") | F.col("Date").endswith(" 12:00:00 PM")).count()
ids_repetidos = filas - df_raw.select("ID").distinct().count()
casos_repetidos = df_raw.groupBy("Case Number").count().filter("count > 1").count()
anios = sorted(r[0] for r in df_raw.select("Year").distinct().collect())
pct_arresto = df_raw.filter(F.col("Arrest")).count() * 100 / filas
pct_domestico = df_raw.filter(F.col("Domestic")).count() * 100 / filas
pct_top2 = top_tipos["% del total"].head(2).sum()
fecha = F.try_to_timestamp("Date", F.lit("MM/dd/yyyy hh:mm:ss a"))
fechas = df_raw.select(F.date_format(F.min(fecha), "dd/MM/yyyy"), F.date_format(F.max(fecha), "dd/MM/yyyy")).first()
ward_cero = df_raw.filter(F.col("Ward") == 0).count()
area_cero = df_raw.filter(F.col("Community Area") == 0).count()
distritos_raros = (df_raw.groupBy("District").count().filter("count < 1000")
                   .orderBy("District").collect())   # los 22 distritos regulares tienen miles de registros cada uno
n = distintos["valores distintos"]

observaciones = {
    "ID": f"Identificador único del incidente; {ids_repetidos} repetidos.",
    "Case Number": f"Número de caso policial; {casos_repetidos} se repiten: varios incidentes de un mismo caso, no errores de carga (Fase 2).",
    "Date": f"Fecha y hora. Spark la lee como texto (MM/dd/yyyy hh:mm:ss AM/PM); se convierte en A2. Rango: {fechas[0]} a {fechas[1]}. "
            f"El {hora_exacta * 100 / filas:.2f} % cae exactamente a medianoche o mediodía: hora anotada cuando no se conoce la real.",
    "Block": "Cuadra aproximada (anonimizada). No se usa en el modelo.",
    "IUCR": f"Código de delito de Illinois ({n['IUCR']} valores). Redundante con Primary Type y Description.",
    "Primary Type": f"Tipo de delito ({n['Primary Type']} categorías): responde el 'qué'. Desbalanceado: las dos más comunes suman {pct_top2:.1f} %.",
    "Description": f"Subtipo de delito ({n['Description']} valores). Redundante con Primary Type para modelar.",
    "Location Description": f"Tipo de lugar ({n['Location Description']} valores). Los vacíos se imputan como DESCONOCIDO en A2.",
    "Arrest": f"Hubo arresto en el {pct_arresto:.1f} % de los casos. Muy alto en delitos detectados por patrullaje (sesgo, Fase 2).",
    "Domestic": f"Violencia doméstica en el {pct_domestico:.1f} % de los casos.",
    "Beat": f"Zona de patrullaje ({n['Beat']} valores): unidad espacial del modelo. Leída como número, pierde el cero inicial (0931 → 931).",
    "District": f"Distrito policial ({n['District']} valores). Códigos poco frecuentes: "
                + ", ".join(f"{r['District']} ({r['count']} registros)" for r in distritos_raros) + "; se revisan en A2.",
    "Ward": f"Distrito electoral ({n['Ward']} valores). {ward_cero} registros con código 0, que no existe. No se usa en el modelo.",
    "Community Area": f"Área comunitaria ({n['Community Area']} valores): nivel de las gráficas por zona. {area_cero} registros con código 0 (las áreas van de 1 a 77); se revisan en A2.",
    "FBI Code": f"Clasificación del FBI ({n['FBI Code']} valores). Redundante con Primary Type.",
    "X Coordinate": "Coordenada en plano métrico (State Plane de Illinois, en pies). Se conserva para el clustering (Fase 2); los vacíos se imputan con el centro del beat en A2.",
    "Y Coordinate": "Igual que X Coordinate.",
    "Year": f"Año del incidente; valores: {anios}. Constante: se elimina en A2.",
    "Updated On": "Fecha de la última actualización en el portal. Metadato que filtraría información posterior al hecho: se elimina en A2.",
    "Latitude": "Coordenada geográfica. Mismos vacíos que X/Y.",
    "Longitude": "Coordenada geográfica. Mismos vacíos que X/Y.",
    "Location": "Punto 'POINT (lon lat)', redundante con Latitude y Longitude: se elimina en A2.",
}

tabla_variables = pd.DataFrame([
    {"variable": c, "tipo": t, "nulos": nulos[c], "% nulos": round(nulos[c] * 100 / filas, 2), "observaciones": observaciones[c]}
    for c, t in df_raw.dtypes
])
tabla_variables

,variable,tipo,nulos,% nulos,observaciones
0,ID,int,0,0.00,Identificador único del incidente; 0 repetidos.
1,Case Number,string,0,0.00,"Número de caso policial; 18 se repiten: varios incidentes de un mismo caso, no errores de carga (Fase 2)."
2,Date,string,0,0.00,Fecha y hora. Spark la lee como texto (MM/dd/yyyy hh:mm:ss AM/PM); se convierte en A2. Rango: 01/01/2025 a 31/12/2025. El 6.12 % cae exactamente a medianoche o mediodía: hora anotada cuando no se conoce la real.
3,Block,string,0,0.00,Cuadra aproximada (anonimizada). No se usa en el modelo.
4,IUCR,string,0,0.00,Código de delito de Illinois (337 valores). Redundante con Primary Type y Description.
5,Primary Type,string,0,0.00,Tipo de delito (31 categorías): responde el 'qué'. Desbalanceado: las dos más comunes suman 41.1 %.
6,Description,string,0,0.00,Subtipo de delito (315 valores). Redundante con Primary Type para modelar.
7,Location Description,string,1234,0.52,Tipo de lugar (130 valores). Los vacíos se imputan como DESCONOCIDO en A2.
8,Arrest,boolean,0,0.00,"Hubo arresto en el 16.1 % de los casos. Muy alto en delitos detectados por patrullaje (sesgo, Fase 2)."
9,Domestic,boolean,0,0.00,Violencia doméstica en el 19.0 % de los casos.


### Hallazgos de A1

- **Tamaño y cobertura.** 238,174 incidentes y 22 columnas, del 1 de enero al 31 de diciembre de 2025, leídos desde HDFS (ingesta del 23/10/2026). El conteo coincide con el que reportaba la API del portal el día de la descarga.
- **Pocos nulos.** 8 columnas tienen vacíos y ninguna pasa del 1.2 %: las cinco de ubicación (`X Coordinate`, `Y Coordinate`, `Latitude`, `Longitude` y `Location`, 2,690 registros · 1.13 %), `Location Description` (1,234 · 0.52 %), `Community Area` (9) y `Ward` (1).
- **Identificadores.** Ningún `ID` se repite. Los 18 `Case Number` repetidos son varios incidentes de un mismo caso, no errores de carga. Los duplicados exactos se revisan en A2.
- **Hora imputada.** El 6.12 % de los registros tiene la hora exacta a medianoche o mediodía, la que se anota cuando no se conoce la real. Se marca en A2 para no usarla en los análisis por hora.
- **Códigos inválidos.** 321 registros tienen `Ward` y `Community Area` en 0 (no existen esas zonas) y hay dos distritos fuera de los 22 regulares: 31 (20 registros) y 61 (313). Su tratamiento se decide en A2.
- **Clases desbalanceadas.** Hay 31 tipos de delito; THEFT (23.18 %) y BATTERY (17.91 %) suman el 41.1 % de los incidentes.
- **Tipos inferidos.** Spark lee `Date` como texto y los códigos geográficos (`Beat`, `District`, `Ward`, `Community Area`) como números; los tipos definitivos se fijan en A2.
- **Columnas que se eliminan en A2:** `Year` (constante), `Location` (redundante con latitud y longitud) y `Updated On` (información posterior al hecho). `IUCR`, `Description` y `FBI Code` se conservan, pero no entran al modelo por ser redundantes con `Primary Type`.

---

## Cierre

In [9]:
spark.stop()   # libera la sesión: en la instancia solo cabe una sesión de Spark a la vez